In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import xgboost as xgb
from scipy.optimize import differential_evolution
import warnings
warnings.filterwarnings('ignore')

# Load & prepare data 
df = pd.read_excel('Dynamic Tests Final Data for ML.xlsx', sheet_name='Sheet1')
df = df.dropna().head(180)

features = ['Ratio', 'Cement', 'Length (mm)', 'Diameter (mm)', 'Strain Rate (/s)', 
            'Shape (k)', 'Scale (λ)', 'Mean (mm)', 'Std (mm)', 'Porosity', 'Density (g/cc)']
X = df[features]
targets = ['Peak Stress (MPa)', 'Modulus (GPa)', 'Vp (m/s)', 'St Compressive Strength (MPa)']
y = df[targets]

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Dataset ready:", X_train.shape, "train,", X_test.shape, "test")

# HYPERPARAMETER FUNCTIONS
def rf_objective(params):
    n_estimators, max_depth, min_samples_split = int(params[0]), int(params[1]), int(params[2])
    model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, 
                                  min_samples_split=min_samples_split, random_state=42)
    model.fit(X_train, y_train.iloc[:, 0])
    pred = model.predict(X_test)
    return -r2_score(y_test.iloc[:, 0], pred)  # Minimize negative R²

def svr_objective(params):
    C, epsilon, gamma = params[0], params[1], params[2]
    model = SVR(kernel='rbf', C=C, epsilon=epsilon, gamma=gamma)
    model.fit(X_train, y_train.iloc[:, 0])
    pred = model.predict(X_test)
    return -r2_score(y_test.iloc[:, 0], pred)

def bpnn_objective(params):
    hidden_layer_sizes, alpha, learning_rate = int(params[0]), params[1], params[2]
    model = MLPRegressor(hidden_layer_sizes=(int(hidden_layer_sizes),), alpha=alpha,
                        learning_rate_init=learning_rate, max_iter=1000, random_state=42)
    model.fit(X_train, y_train.iloc[:, 0])
    pred = model.predict(X_test)
    return -r2_score(y_test.iloc[:, 0], pred)

def xgb_objective(params):
    n_estimators, max_depth, learning_rate, subsample = params
    model = xgb.XGBRegressor(n_estimators=int(n_estimators), max_depth=int(max_depth),
                           learning_rate=learning_rate, subsample=subsample, random_state=42)
    model.fit(X_train, y_train.iloc[:, 0])
    pred = model.predict(X_test)
    return -r2_score(y_test.iloc[:, 0], pred)

# GENETIC ALGORITHM HYPERTUNING 
print("\n🔧 HYPERPARAMETER TUNING (GA-style)...")

# Random Forest - GA bounds [n_estimators, max_depth, min_samples_split]
rf_bounds = [(50, 200), (3, 15), (2, 20)]
rf_result = differential_evolution(rf_objective, rf_bounds, maxiter=20, popsize=15)
rf_best = {'n_estimators': int(rf_result.x[0]), 'max_depth': int(rf_result.x[1]), 
           'min_samples_split': int(rf_result.x[2])}
print(f"RF Best: {rf_best}, R²: {-rf_result.fun:.3f}")

# SVR - GA bounds [C, epsilon, gamma]
svr_bounds = [(0.1, 100), (0.01, 1), (0.001, 1)]
svr_result = differential_evolution(svr_objective, svr_bounds, maxiter=20, popsize=15)
svr_best = {'C': svr_result.x[0], 'epsilon': svr_result.x[1], 'gamma': svr_result.x[2]}
print(f"SVR Best: {svr_best}, R²: {-svr_result.fun:.3f}")

# BPNN - GA bounds [hidden_size, alpha, learning_rate]
bpnn_bounds = [(50, 200), (0.0001, 0.1), (0.001, 0.1)]
bpnn_result = differential_evolution(bpnn_objective, bpnn_bounds, maxiter=20, popsize=15)
bpnn_best = {'hidden': int(bpnn_result.x[0]), 'alpha': bpnn_result.x[1], 'lr': bpnn_result.x[2]}
print(f"BPNN Best: {bpnn_best}, R²: {-bpnn_result.fun:.3f}")

# XGBoost - GA bounds [n_estimators, max_depth, learning_rate, subsample]
xgb_bounds = [(50, 200), (3, 10), (0.01, 0.3), (0.6, 1.0)]
xgb_result = differential_evolution(xgb_objective, xgb_bounds, maxiter=20, popsize=15)
xgb_best = {'n_estimators': int(xgb_result.x[0]), 'max_depth': int(xgb_result.x[1]), 
            'learning_rate': xgb_result.x[2], 'subsample': xgb_result.x[3]}
print(f"XGBoost Best: {xgb_best}, R²: {-xgb_result.fun:.3f}")

# TRAIN FINAL MODELS WITH BEST PARAMS 
print("\n FINAL MODELS PERFORMANCE (Peak Stress MPa):")
models = {}

# Random Forest
rf_model = RandomForestRegressor(**rf_best, random_state=42)
rf_model.fit(X_train, y_train.iloc[:, 0])
rf_pred = rf_model.predict(X_test)
print(f"RF Tuned: R²={r2_score(y_train.iloc[:, 0], rf_model.predict(X_train)):.3f}/{r2_score(y_test.iloc[:, 0], rf_pred):.3f}")

# SVR  
svr_model = SVR(kernel='rbf', **svr_best)
svr_model.fit(X_train, y_train.iloc[:, 0])
svr_pred = svr_model.predict(X_test)
print(f"SVR Tuned: R²={r2_score(y_train.iloc[:, 0], svr_model.predict(X_train)):.3f}/{r2_score(y_test.iloc[:, 0], svr_pred):.3f}")

# BPNN 
bpnn_model = MLPRegressor(hidden_layer_sizes=(bpnn_best['hidden'],), alpha=bpnn_best['alpha'],
                         learning_rate_init=bpnn_best['lr'], max_iter=2000, random_state=42)
bpnn_model.fit(X_train, y_train.iloc[:, 0])
bpnn_pred = bpnn_model.predict(X_test)
print(f"BPNN Tuned: R²={r2_score(y_train.iloc[:, 0], bpnn_model.predict(X_train)):.3f}/{r2_score(y_test.iloc[:, 0], bpnn_pred):.3f}")

# XGBoost 
xgb_model = xgb.XGBRegressor(**xgb_best, random_state=42)
xgb_model.fit(X_train, y_train.iloc[:, 0])
xgb_pred = xgb_model.predict(X_test)
print(f"XGBoost: R²={r2_score(y_train.iloc[:, 0], xgb_model.predict(X_train)):.3f}/{r2_score(y_test.iloc[:, 0], xgb_pred):.3f}")

models = {'RF': rf_model, 'SVR': svr_model, 'BPNN': bpnn_model, 'XGB': xgb_model}

# RANKING 
results = pd.DataFrame({
    'Model': ['Random Forest', 'SVR', 'BPNN', 'XGBoost'],
    'Train_R2': [r2_score(y_train.iloc[:, 0], rf_model.predict(X_train)),
                 r2_score(y_train.iloc[:, 0], svr_model.predict(X_train)),
                 r2_score(y_train.iloc[:, 0], bpnn_model.predict(X_train)),
                 r2_score(y_train.iloc[:, 0], xgb_model.predict(X_train))],
    'Test_R2': [r2_score(y_test.iloc[:, 0], rf_pred),
                r2_score(y_test.iloc[:, 0], svr_pred),
                r2_score(y_test.iloc[:, 0], bpnn_pred),
                r2_score(y_test.iloc[:, 0], xgb_pred)]
})
print("\n FINAL RANKING (Test R²):")
print(results.sort_values('Test_R2', ascending=False).round(3))


Dataset ready: (132, 11) train, (33, 11) test

🔧 HYPERPARAMETER TUNING (GA-style)...
RF Best: {'n_estimators': 195, 'max_depth': 3, 'min_samples_split': 14}, R²: 0.842
SVR Best: {'C': np.float64(72.53950993910253), 'epsilon': np.float64(0.21787972289876578), 'gamma': np.float64(0.0024403849617887774)}, R²: 0.868
BPNN Best: {'hidden': 86, 'alpha': np.float64(0.040986213915071325), 'lr': np.float64(0.0015882690915777825)}, R²: 0.799
XGBoost Best: {'n_estimators': 54, 'max_depth': 3, 'learning_rate': np.float64(0.04642844789030559), 'subsample': np.float64(0.9726539134859)}, R²: 0.850

 FINAL MODELS PERFORMANCE (Peak Stress MPa):
RF Tuned: R²=0.803/0.842
SVR Tuned: R²=0.691/0.868
BPNN Tuned: R²=0.942/0.749
XGBoost: R²=0.875/0.850

 FINAL RANKING (Test R²):
           Model  Train_R2  Test_R2
1            SVR     0.691    0.868
3        XGBoost     0.875    0.850
0  Random Forest     0.803    0.842
2           BPNN     0.942    0.749
